In [ ]:
%load_ext autoreload
%autoreload 2
import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
from app.logger import *
import json5,json
import fitz #type: ignore

from app.insur.fund_data import *
from app.utils import *
from app.konstant import get_config, get_regex

from app.amc.fund_data import *

utils = Helper()

In [ ]:
#INSURANCE FUND
amc_id = '81_0'
path = r"81_30-Apr-26_IF.pdf"
config = get_config("2026",amc_id)
regex = get_regex("2026")

object = GeneraliLifeINSR(config,regex,path)
title,path_pdf= object.check_and_highlight(path)
# print("done")
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

In [ ]:
# object = GeneraliLifeINSR(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)


In [ ]:
save_path = os.path.join(object.JSON_PATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

In [ ]:
pattern = "([A-Z]{1}[a-z]+\\s*[A-Z]{1}[a-z]+)"
for fund, content in final_text.items():
    # check = 'before.fund_manager'
    for key in content:
        if key.endswith("fund_manager_details"):
            print(fund)
            text =re.sub("[^A-Za-z0-9\\s\\-\\(\\)\\.\\,\\+\\%\\:\\&]+", "",content[key]).strip()
            print(text)
            match = re.findall(pattern,text, re.IGNORECASE)
            print(match)